In [33]:
import pandas as pd
import numpy as np
import pickle
from tensorflow.keras.models import load_model
import sklearn
from sklearn.preprocessing import LabelEncoder
import joblib


In [34]:


df_embed = pd.read_csv("/Users/vaidehipandya/Desktop/Work/Heatmap/training_data_embeddings_only.csv")
tile_coords = pd.read_csv("/Users/vaidehipandya/Desktop/Work/Heatmap/TCGA_55_7574_final_tile_coordinates.csv")


model_path = "/Users/vaidehipandya/Desktop/Work/Heatmap/hierarchical_classifier_best.keras"
encoder = "/Users/vaidehipandya/Desktop/Work/Heatmap/label_encoders.pkl"
label_col = "/Users/vaidehipandya/Desktop/Work/Heatmap/label_columns.pkl"

annotation_labels = "/Users/vaidehipandya/Desktop/Work/Heatmap/HPC_consensus_labels_clean.csv"
heatmap_table= "/Users/vaidehipandya/Desktop/Work/Heatmap/heatmap_table.csv"

output = "/Users/vaidehipandya/Desktop/Work/Heatmap/tile_hpc_probabilities.csv"

In [35]:
df_spatial = tile_coords[[
    "slide_tile",
    "x_native",
    "y_native",
    "slides"   # optional but useful
]].copy()

In [36]:
df_heatmap = df_spatial.merge(
    df_embed,
    on="slide_tile",
    how="inner"
)

In [37]:
df_heatmap.to_csv(heatmap_table, index=False)

In [38]:
z_cols = [c for c in df_heatmap.columns if c.startswith("z")]
X = df_heatmap[z_cols].values

In [39]:
model = load_model(model_path)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 512)       │     66,048 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 512)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │    131,328 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cluster_homogeneity │ (None, 2)         │        514 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ malignant_label     │ (None, 2)         │        514 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inflammation_label  │ (None, 3)         │        771 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ necrosis_label      │ (None, 3)         │        771 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stromal_epithelium… │ (None, 4)         │      1,028 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stromal_cellularity │ (None, 4)         │      1,028 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ malignant_subtype   │ (None, 6)         │      1,542 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ second_malignant_s… │ (None, 7)         │      1,799 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ non_malignant_subt… │ (None, 10)        │      2,570 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ second_non_maligna… │ (None, 7)         │      1,799 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 629,138 (2.40 MB)

 Trainable params: 209,712 (819.19 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 419,426 (1.60 MB)

In [40]:
preds = model.predict(X, batch_size=256, verbose=1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step


In [41]:
print(type(preds))
print(len(preds))
print(preds[0].shape)

<class 'list'>
10
(254, 2)


In [42]:
import joblib
import numpy as np

label_cols = joblib.load(label_col)   # list of head names in correct order

if not isinstance(preds, list):
    preds = [preds]

assert len(preds) == len(label_cols), f"preds={len(preds)} but label_cols={len(label_cols)}"

head_to_probs = {label_cols[i]: preds[i] for i in range(len(label_cols))}

print("Heads in predictions:", list(head_to_probs.keys()))
for h in label_cols:
    print(h, head_to_probs[h].shape)


Heads in predictions: ['cluster_homogeneity', 'malignant_label', 'inflammation_label', 'necrosis_label', 'stromal_epithelium_ratio', 'stromal_cellularity', 'malignant_subtype', 'second_malignant_subtype', 'non_malignant_subtype', 'second_non_malignant_subtype']
cluster_homogeneity (254, 2)
malignant_label (254, 2)
inflammation_label (254, 3)
necrosis_label (254, 3)
stromal_epithelium_ratio (254, 4)
stromal_cellularity (254, 4)
malignant_subtype (254, 6)
second_malignant_subtype (254, 7)
non_malignant_subtype (254, 10)
second_non_malignant_subtype (254, 7)


In [43]:
encoders = joblib.load(encoder)  # dict head -> LabelEncoder

# df_tile_labels = pd.read_csv(tile_label)
# print(df_tile_labels.columns)

df_hpc_profile = pd.read_csv(annotation_labels)

# Basic cleaning
df_hpc_profile["hpc_id"] = pd.to_numeric(df_hpc_profile["hpc_id"], errors="coerce")
df_hpc_profile = df_hpc_profile.dropna(subset=["hpc_id"]).copy()
df_hpc_profile["hpc_id"] = df_hpc_profile["hpc_id"].astype(int)

In [44]:
# Ensure preds is list aligned to label_cols
if not isinstance(preds, list):
    preds = [preds]
assert len(preds) == len(label_cols)

# Build predicted label table
df_pred = df_heatmap[["slide_tile"]].copy()

for i, head in enumerate(label_cols):
    probs = preds[i]                      # shape: (n_tiles, n_classes)
    idx = np.argmax(probs, axis=1)        # predicted class index per tile

    # Convert index to label string using the saved encoder
    # encoders[head] is a LabelEncoder
    df_pred[head] = encoders[head].inverse_transform(idx)

    # Optional confidence column per head
    df_pred[head + "_conf"] = probs[np.arange(len(idx)), idx]

In [45]:
def clean_label(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s == "":
        return None
    return s



In [46]:
# preds might be np.array if 1 head, force list
if not isinstance(preds, list):
    preds = [preds]

assert len(preds) == len(label_cols), f"preds={len(preds)} heads but label_cols={len(label_cols)}"

# build df_pred_labels: slide_tile + predicted phenotype labels
df_pred_labels = df_heatmap[["slide_tile"]].copy()

for i, head in enumerate(label_cols):
    probs = preds[i]  # (n_tiles, n_classes)
    idx = np.argmax(probs, axis=1)
    classes = encoders[head].classes_  # label strings in training order
    df_pred_labels[head] = classes[idx].astype(str)

print(df_pred_labels.head())

                           slide_tile cluster_homogeneity malignant_label  \
0  TCGA-55-7574-01Z-00-DX1_18_15.jpeg        high (> 75%)              no   
1   TCGA-55-7574-01Z-00-DX1_14_6.jpeg        high (> 75%)             yes   
2  TCGA-55-7574-01Z-00-DX1_11_23.jpeg        high (> 75%)             yes   
3  TCGA-55-7574-01Z-00-DX1_10_20.jpeg        high (> 75%)             yes   
4   TCGA-55-7574-01Z-00-DX1_23_1.jpeg        high (> 75%)              no   

  inflammation_label necrosis_label stromal_epithelium_ratio  \
0             marked           none           not_applicable   
1      mild-moderate           none            roughly equal   
2        none-sparse           none              more stroma   
3      mild-moderate           none            roughly equal   
4        none-sparse           none           not_applicable   

  stromal_cellularity malignant_subtype second_malignant_subtype  \
0      not_applicable    not_applicable           not_applicable   
1               

In [47]:
# df_hpc_profile = build_hpc_profile_from_tiles(
#     df_pred2,        # IMPORTANT: df_pred2, not df_tile_labels
#     label_cols,
#     min_mode_frac=0.4
# )

print("HPC profile shape:", df_hpc_profile.shape)
print(df_hpc_profile.head(20))
# -----------------------
# Save to CSV for inspection
# -----------------------
OUT_HPC_PROFILE = "/Users/vaidehipandya/Desktop/Work/Heatmap/hpc_auto_profiles.csv"

df_hpc_profile.to_csv(OUT_HPC_PROFILE, index=False)

print("Saved HPC auto profile →", OUT_HPC_PROFILE)

HPC profile shape: (71, 11)
    hpc_id malignant_label malignant_subtype second_malignant_subtype  \
0        0             Yes            Acinar                  Lepidic   
1        1              No               NaN                      NaN   
2        2              No               NaN                      NaN   
3        3              No               NaN                      NaN   
4        4              No               NaN                      NaN   
5        5             Yes            Acinar           Micropapillary   
6        6              No               NaN                      NaN   
7        7              No               NaN                      NaN   
8        8              No               NaN                      NaN   
9        9              No               NaN                      NaN   
10      10              No               NaN                      NaN   
11      11             Yes            Acinar                Papillary   
12      12             

In [48]:
print("df_hpc_profile columns:", df_hpc_profile.columns.tolist())
print("label_cols:", label_cols)

# which label cols are present in df_tile_labels?
present = [c for c in label_cols if c in df_hpc_profile.columns]
print("label cols present in df_tile_labels:", present)

df_hpc_profile columns: ['hpc_id', 'malignant_label', 'malignant_subtype', 'second_malignant_subtype', 'non_malignant_subtype', 'second_non_malignant_subtype', 'inflammation_label', 'necrosis_label', 'stromal_epithelium_ratio', 'stromal_cellularity', 'cluster_homogeneity']
label_cols: ['cluster_homogeneity', 'malignant_label', 'inflammation_label', 'necrosis_label', 'stromal_epithelium_ratio', 'stromal_cellularity', 'malignant_subtype', 'second_malignant_subtype', 'non_malignant_subtype', 'second_non_malignant_subtype']
label cols present in df_tile_labels: ['cluster_homogeneity', 'malignant_label', 'inflammation_label', 'necrosis_label', 'stromal_epithelium_ratio', 'stromal_cellularity', 'malignant_subtype', 'second_malignant_subtype', 'non_malignant_subtype', 'second_non_malignant_subtype']


In [49]:
import numpy as np
import pandas as pd
import joblib

EPS = 1e-9



def softmax_np(x, axis=1):
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / (np.sum(ex, axis=axis, keepdims=True) + EPS)

def safe_log(p):
    return np.log(np.clip(p, EPS, 1.0))

def build_class_index(encoders, head_name):
    classes = encoders[head_name].classes_
    return {str(c): i for i, c in enumerate(classes)}

In [ ]:
# import pandas as pd
# import numpy as np



# 3) Heads used for scoring (ALL important heads, but treat second_* as alternates not separate heads)
# candidate_profile_cols = [
#     "cluster_homogeneity",
#     "malignant_label",
#     "inflammation_label",
#     "necrosis_label",
#     "stromal_epithelium_ratio",
#     "stromal_cellularity",
#     "malignant_subtype",
#     "non_malignant_subtype",
# ]

# print("Profile cols used:", profile_cols)

# # 4) Diagnostics: show annotation labels that do not exist in encoders (helps catch typos)
# for head in profile_cols:
#     ann_vals = set(df_hpc_profile[head].dropna().astype(str).str.strip().unique())
#     enc_vals = set(map(str, encoders[head].classes_))
#     missing = ann_vals - enc_vals
#     if missing:
#         print("WARNING annotation labels not in encoder for head:", head)
#         print("Examples:", list(sorted(missing))[:20])

# # Also check second subtype columns if present
# for head, colname in [("malignant_subtype", "second_malignant_subtype"),
#                       ("non_malignant_subtype", "second_non_malignant_subtype")]:
#     if colname in df_hpc_profile.columns and head in encoders:
#         ann_vals = set(df_hpc_profile[colname].dropna().astype(str).str.strip().unique())
#         enc_vals = set(map(str, encoders[head].classes_))
#         missing = ann_vals - enc_vals
#         if missing:
#             print("WARNING annotation labels not in encoder for column:", colname, "(uses encoder of", head, ")")
#             print("Examples:", list(sorted(missing))[:20])

# # 5) Filter out HPC rows with too little usable information (keeps scoring stable)
# #    Here we count filled labels across: malignant_label + other heads + subtype (and optional second subtype)
# def count_filled_labels(row):
#     n = 0
#     for h in profile_cols:
#         if clean_label(row.get(h, None)) is not None:
#             n += 1
#     if "second_malignant_subtype" in df_hpc_profile.columns:
#         if clean_label(row.get("second_malignant_subtype", None)) is not None:
#             n += 1
#     if "second_non_malignant_subtype" in df_hpc_profile.columns:
#         if clean_label(row.get("second_non_malignant_subtype", None)) is not None:
#             n += 1
#     return n

# df_hpc_profile["n_filled"] = df_hpc_profile.apply(count_filled_labels, axis=1)

# MIN_LABELS = 2
# df_hpc_profile = df_hpc_profile[df_hpc_profile["n_filled"] >= MIN_LABELS].copy()
# print("HPCs after filtering:", df_hpc_profile["hpc_id"].nunique())

# # 6) Build label->index dict per head
# head_to_class_index = {h: build_class_index(encoders, h) for h in profile_cols}

# # 7) Prepare HPC list
# hpc_ids = sorted(df_hpc_profile["hpc_id"].unique().tolist())
# hpc_to_col = {h: j for j, h in enumerate(hpc_ids)}

# n_tiles = df_heatmap.shape[0]
# n_hpcs = len(hpc_ids)

# print("Tiles:", n_tiles, "HPCs scored:", n_hpcs)

# # 8) Compute log scores: shape (n_tiles, n_hpcs)
# log_scores = np.zeros((n_tiles, n_hpcs), dtype=np.float32)
# df_hpc_indexed = df_hpc_profile.set_index("hpc_id")

# for hpc in hpc_ids:
#     row = df_hpc_indexed.loc[hpc]
#     score = np.zeros((n_tiles,), dtype=np.float32)

#     # Optional guard: enforce malignant vs non-malignant branch consistency based on malignant_label
#     # Only applied if malignant_label exists in scoring heads
#     mal_flag = clean_label(row.get("malignant_label", None))
#     # Normalize some common variants if they exist (keep as-is if you don't have these)
#     if mal_flag is not None:
#         mf = mal_flag.lower()
#         if mf in ["yes", "malignant", "true", "1"]:
#             mal_flag_norm = "Yes"
#         elif mf in ["no", "non-malignant", "non malignant", "false", "0"]:
#             mal_flag_norm = "No"
#         else:
#             mal_flag_norm = mal_flag
#     else:
#         mal_flag_norm = None

#     for head in profile_cols:
#         # Branch consistency: if HPC is malignant, do not score non-malignant subtype; vice versa
#         if mal_flag_norm == "Yes" and head == "non_malignant_subtype":
#             continue
#         if mal_flag_norm == "No" and head == "malignant_subtype":
#             continue

#         probs_head = head_to_probs[head]
#         class_index = head_to_class_index[head]

#         label1 = clean_label(row.get(head, None))
#         if label1 is None:
#             continue

#         # Subtype handling: allow 2 labels (primary OR second), take max prob once
#         if head == "malignant_subtype" and "second_malignant_subtype" in df_hpc_profile.columns:
#             label2 = clean_label(row.get("second_malignant_subtype", None))

#             p1 = probs_head[:, class_index[label1]] if label1 in class_index else None
#             p2 = probs_head[:, class_index[label2]] if (label2 and label2 in class_index) else None

#             if p1 is None and p2 is None:
#                 continue
#             if p1 is None:
#                 p = p2
#             elif p2 is None:
#                 p = p1
#             else:
#                 p = np.maximum(p1, p2)

#             score += safe_log(p)
#             continue

#         if head == "non_malignant_subtype" and "second_non_malignant_subtype" in df_hpc_profile.columns:
#             label2 = clean_label(row.get("second_non_malignant_subtype", None))

#             p1 = probs_head[:, class_index[label1]] if label1 in class_index else None
#             p2 = probs_head[:, class_index[label2]] if (label2 and label2 in class_index) else None

#             if p1 is None and p2 is None:
#                 continue
#             if p1 is None:
#                 p = p2
#             elif p2 is None:
#                 p = p1
#             else:
#                 p = np.maximum(p1, p2)

#             score += safe_log(p)
#             continue

#         # Normal case: single label
#         if label1 not in class_index:
#             continue

#         score += safe_log(probs_head[:, class_index[label1]])

#     log_scores[:, hpc_to_col[hpc]] = score

# print("log_scores min max:", float(log_scores.min()), float(log_scores.max()))

# # 9) Convert scores to per tile probability over HPC ids
# T = 2.0  # try 1.5, 2.0, 3.0
# hpc_probs = softmax_np(log_scores / T, axis=1)

# for T in [1.0, 1.5, 2.0, 3.0]:
#     hpc_probs = softmax_np(log_scores / T, axis=1)
#     top_idx = np.argmax(hpc_probs, axis=1)
#     top_prob = hpc_probs[np.arange(n_tiles), top_idx]
#     print("T", T, "mean top prob", float(top_prob.mean()), "median", float(np.median(top_prob)))

# top_idx = np.argmax(hpc_probs, axis=1)
# top_hpc = [hpc_ids[i] for i in top_idx]
# top_prob = hpc_probs[np.arange(n_tiles), top_idx]

# # 10) Build final wide table for Streamlit
# keep_cols = [c for c in ["slides", "tiles", "slide_tile", "x_native", "y_native"] if c in df_heatmap.columns]
# out = df_heatmap[keep_cols].copy()

# out["hpc_top1"] = top_hpc
# out["hpc_top1_prob"] = top_prob

# for j, hpc in enumerate(hpc_ids):
#     out[f"p_hpc_{hpc}"] = hpc_probs[:, j]

# OUT_CSV = "/Users/vaidehipandya/Desktop/Work/Heatmap/tile_hpc_probabilities_wide.csv"
# out.to_csv(OUT_CSV, index=False)

# print("Saved:", OUT_CSV)
# print(out.head(5))

# # Optional sanity check: probabilities should sum to ~1
# prob_cols = [c for c in out.columns if c.startswith("p_hpc_")]
# print("Max deviation from 1:", float((out[prob_cols].sum(axis=1) - 1).abs().max()))

In [50]:
import pandas as pd

print("========== ENCODER vs ANNOTATION COMPARISON ==========\n")

for head in profile_cols:

    print("\n============================================")
    print("HEAD:", head)
    print("============================================")

    # Encoder labels (what model understands)
    encoder_labels = list(map(str, encoders[head].classes_))
    print("\nEncoder classes:")
    for e in encoder_labels:
        print("  -", e)

    # Annotation labels (what your CSV contains)
    ann_labels = (
        df_hpc_profile[head]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )

    print("\nRaw annotation labels:")
    for a in ann_labels:
        print("  -", a)

    # Quick direct match check (exact string match)
    exact_matches = [a for a in ann_labels if a in encoder_labels]
    not_matching = [a for a in ann_labels if a not in encoder_labels]

    print("\nExact matches with encoder:")
    if exact_matches:
        for m in exact_matches:
            print("  ✓", m)
    else:
        print("  None")

    print("\nAnnotation labels NOT matching encoder:")
    if not_matching:
        for m in not_matching:
            print("  ✗", m)
    else:
        print("  None")

    # Also check second subtype columns if relevant
    if head == "malignant_subtype" and "second_malignant_subtype" in df_hpc_profile.columns:
        second_vals = (
            df_hpc_profile["second_malignant_subtype"]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()
            .tolist()
        )
        print("\nSecond malignant subtype annotation values:")
        for s in second_vals:
            print("  -", s)

    if head == "non_malignant_subtype" and "second_non_malignant_subtype" in df_hpc_profile.columns:
        second_vals = (
            df_hpc_profile["second_non_malignant_subtype"]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()
            .tolist()
        )
        print("\nSecond non-malignant subtype annotation values:")
        for s in second_vals:
            print("  -", s)

print("\n========== END REPORT ==========")

========== ENCODER vs ANNOTATION COMPARISON ==========


HEAD: cluster_homogeneity

Encoder classes:
  - high (> 75%)
  - moderate (50 - 75%)

Raw annotation labels:
  - High (> 75%)
  - Moderate (50 - 75%)

Exact matches with encoder:
  None

Annotation labels NOT matching encoder:
  ✗ High (> 75%)
  ✗ Moderate (50 - 75%)

HEAD: malignant_label

Encoder classes:
  - no
  - yes

Raw annotation labels:
  - Yes
  - No

Exact matches with encoder:
  None

Annotation labels NOT matching encoder:
  ✗ Yes
  ✗ No

HEAD: inflammation_label

Encoder classes:
  - marked
  - mild-moderate
  - none-sparse

Raw annotation labels:
  - Mild-moderate
  - None-sparse
  - Marked

Exact matches with encoder:
  None

Annotation labels NOT matching encoder:
  ✗ Mild-moderate
  ✗ None-sparse
  ✗ Marked

HEAD: necrosis_label

Encoder classes:
  - none
  - some
  - universal

Raw annotation labels:
  - None
  - Some
  - Universal

Exact matches with encoder:
  None

Annotation labels NOT matching encoder:
  ✗

In [51]:
import pandas as pd
import numpy as np

# # Heads you score
# profile_cols = [
#     "cluster_homogeneity",
#     "malignant_label",
#     "inflammation_label",
#     "necrosis_label",
#     "stromal_epithelium_ratio",
#     "stromal_cellularity",
#     "malignant_subtype",
#     "non_malignant_subtype",
# ]

# Optional columns used as alternates
SECOND_MALIGNANT = "second_malignant_subtype"
SECOND_NONMALIGNANT = "second_non_malignant_subtype"

def normalize_to_encoder(x, head, encoders):
    """
    Convert annotation cell -> encoder class label
    Rules:
      - blank/NaN -> 'not_applicable' if encoder supports it, else NaN
      - otherwise lowercase + strip
      - validate against encoder.classes_
      - if not in encoder.classes_, return NaN (or 'not_applicable' if available)
    """
    enc_set = set(map(str, encoders[head].classes_))

    # blank -> not_applicable (if that class exists)
    if pd.isna(x) or str(x).strip() == "":
        return "not_applicable" if "not_applicable" in enc_set else np.nan

    s = str(x).strip().lower()

    # if exact match after lowering
    if s in enc_set:
        return s

    # fallback: if unknown, push to not_applicable if it exists
    return "not_applicable" if "not_applicable" in enc_set else np.nan

# df_hpc_profile is your annotation dataframe loaded from CSV
df_hpc_profile = pd.read_csv(annotation_labels)

# Ensure hpc_id is int
df_hpc_profile["hpc_id"] = pd.to_numeric(df_hpc_profile["hpc_id"], errors="coerce")
df_hpc_profile = df_hpc_profile.dropna(subset=["hpc_id"]).copy()
df_hpc_profile["hpc_id"] = df_hpc_profile["hpc_id"].astype(int)

# Create mapped columns
for head in profile_cols:
    df_hpc_profile[head + "_mapped"] = df_hpc_profile[head].apply(lambda v: normalize_to_encoder(v, head, encoders))

# Map second subtype columns using same encoders as their primary subtype head
if SECOND_MALIGNANT in df_hpc_profile.columns:
    df_hpc_profile[SECOND_MALIGNANT + "_mapped"] = df_hpc_profile[SECOND_MALIGNANT].apply(
        lambda v: normalize_to_encoder(v, "malignant_subtype", encoders)
    )

if SECOND_NONMALIGNANT in df_hpc_profile.columns:
    df_hpc_profile[SECOND_NONMALIGNANT + "_mapped"] = df_hpc_profile[SECOND_NONMALIGNANT].apply(
        lambda v: normalize_to_encoder(v, "non_malignant_subtype", encoders)
    )

# Quick sanity check: show any remaining mismatches for scored heads
for head in profile_cols:
    enc_set = set(map(str, encoders[head].classes_))
    mapped_vals = set(df_hpc_profile[head + "_mapped"].dropna().astype(str).unique())
    bad = mapped_vals - enc_set
    if bad:
        print("Still mismatching after mapping for head:", head, "->", list(bad)[:20])

print("Mapped annotation table ready. Preview:")
cols_preview = ["hpc_id"] + [h + "_mapped" for h in profile_cols]
print(df_hpc_profile[cols_preview].head(10))

Mapped annotation table ready. Preview:
   hpc_id cluster_homogeneity_mapped malignant_label_mapped  \
0       0               high (> 75%)                    yes   
1       1               high (> 75%)                     no   
2       2               high (> 75%)                     no   
3       3               high (> 75%)                     no   
4       4               high (> 75%)                     no   
5       5               high (> 75%)                    yes   
6       6               high (> 75%)                     no   
7       7               high (> 75%)                     no   
8       8               high (> 75%)                     no   
9       9               high (> 75%)                     no   

  inflammation_label_mapped necrosis_label_mapped  \
0             mild-moderate                  none   
1               none-sparse                  none   
2               none-sparse                  none   
3               none-sparse                  none   
4

In [53]:
import pandas as pd
import numpy as np

EPS = 1e-9

def softmax_np(x, axis=1):
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / (np.sum(ex, axis=axis, keepdims=True) + EPS)

def safe_log(p):
    return np.log(np.clip(p, EPS, 1.0))

def clean_label(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s == "":
        return None
    return s

def build_class_index(encoders, head_name):
    classes = encoders[head_name].classes_
    return {str(c): i for i, c in enumerate(classes)}

# -------------------------------------------------------
# REQUIRED INPUTS IN MEMORY
# df_heatmap: must contain slide_tile, x_native, y_native (and optionally slides, tiles)
# preds: model.predict(X)
# encoders: joblib.load("label_encoders.pkl")  -> dict: head -> LabelEncoder
# label_cols: joblib.load("label_columns.pkl") -> list of heads in model output order
# annotation_labels: path to HPC_consensus_labels_clean.csv
# -------------------------------------------------------

# 1) Map preds list to head name -> probs array
if not isinstance(preds, list):
    preds = [preds]

assert len(preds) == len(label_cols), f"preds={len(preds)} heads but label_cols={len(label_cols)}"
head_to_probs = {label_cols[i]: preds[i] for i in range(len(label_cols))}

# 2) Load annotation table as df_hpc_profile
df_hpc_profile = pd.read_csv(annotation_labels)
df_hpc_profile["hpc_id"] = pd.to_numeric(df_hpc_profile["hpc_id"], errors="coerce")
df_hpc_profile = df_hpc_profile.dropna(subset=["hpc_id"]).copy()
df_hpc_profile["hpc_id"] = df_hpc_profile["hpc_id"].astype(int)

# 3) Heads used for scoring
candidate_profile_cols = [
    "cluster_homogeneity",
    "malignant_label",
    "inflammation_label",
    "necrosis_label",
    "stromal_epithelium_ratio",
    "stromal_cellularity",
    "malignant_subtype",
    "non_malignant_subtype",
]

HEAD_W = {
    "cluster_homogeneity": 0.4,
    "malignant_label": 0.6,
    "inflammation_label": 0.5,
    "necrosis_label": 0.5,
    "stromal_epithelium_ratio": 0.4,
    "stromal_cellularity": 0.4,
    "malignant_subtype": 0.6,
    "non_malignant_subtype": 0.6,
}
DEFAULT_W = 0.0
profile_cols = [
    c for c in candidate_profile_cols
    if (c in df_hpc_profile.columns) and (c in head_to_probs) and (c in encoders)
]

print("Profile cols used:", profile_cols)

# -------------------------------------------------------
# 4) MAP ANNOTATION VALUES -> ENCODER COMPATIBLE VALUES
# -------------------------------------------------------
def to_encoder_label(val, head):
    enc_set = set(map(str, encoders[head].classes_))

    # blanks become not_applicable if encoder supports it
    if pd.isna(val) or str(val).strip() == "":
        return "not_applicable" if "not_applicable" in enc_set else None

    s = str(val).strip().lower()
    if s in enc_set:
        return s

    # fallback: unknown labels map to not_applicable if available
    return "not_applicable" if "not_applicable" in enc_set else None

# mapped columns for scoring heads
for head in profile_cols:
    df_hpc_profile[head + "_mapped"] = df_hpc_profile[head].apply(lambda v: to_encoder_label(v, head))
    # w = HEAD_W.get(head, DEFAULT_W)
# mapped columns for second subtype alternates (if present)
if "second_malignant_subtype" in df_hpc_profile.columns:
    df_hpc_profile["second_malignant_subtype_mapped"] = df_hpc_profile["second_malignant_subtype"].apply(
        lambda v: to_encoder_label(v, "malignant_subtype")
    )
else:
    df_hpc_profile["second_malignant_subtype_mapped"] = None

if "second_non_malignant_subtype" in df_hpc_profile.columns:
    df_hpc_profile["second_non_malignant_subtype_mapped"] = df_hpc_profile["second_non_malignant_subtype"].apply(
        lambda v: to_encoder_label(v, "non_malignant_subtype")
    )
else:
    df_hpc_profile["second_non_malignant_subtype_mapped"] = None

# Optional: confirm mapping worked (should be empty sets)
for head in profile_cols:
    enc_set = set(map(str, encoders[head].classes_))
    mapped_vals = set(df_hpc_profile[head + "_mapped"].dropna().astype(str).unique())
    bad = mapped_vals - enc_set
    if bad:
        print("Still mismatching after mapping for head:", head, "->", list(bad)[:20])

# -------------------------------------------------------
# 5) Filter out HPC rows with too little usable information
# -------------------------------------------------------
def count_filled_labels(row):
    n = 0
    for h in profile_cols:
        if clean_label(row.get(h + "_mapped", None)) is not None:
            n += 1
    if clean_label(row.get("second_malignant_subtype_mapped", None)) is not None:
        n += 1
    if clean_label(row.get("second_non_malignant_subtype_mapped", None)) is not None:
        n += 1
    return n

df_hpc_profile["n_filled"] = df_hpc_profile.apply(count_filled_labels, axis=1)

MIN_LABELS = 2
df_hpc_profile = df_hpc_profile[df_hpc_profile["n_filled"] >= MIN_LABELS].copy()
print("HPCs after filtering:", df_hpc_profile["hpc_id"].nunique())

# 6) Build label->index dict per head
head_to_class_index = {h: build_class_index(encoders, h) for h in profile_cols}

# 7) Prepare HPC list
hpc_ids = sorted(df_hpc_profile["hpc_id"].unique().tolist())
hpc_to_col = {h: j for j, h in enumerate(hpc_ids)}

n_tiles = df_heatmap.shape[0]
n_hpcs = len(hpc_ids)
print("Tiles:", n_tiles, "HPCs scored:", n_hpcs)

# 8) Compute log scores: shape (n_tiles, n_hpcs)
log_scores = np.zeros((n_tiles, n_hpcs), dtype=np.float32)
df_hpc_indexed = df_hpc_profile.set_index("hpc_id")

for hpc in hpc_ids:
    row = df_hpc_indexed.loc[hpc]
    score = np.zeros((n_tiles,), dtype=np.float32)

    # malignant_label encoder uses "yes"/"no"
    mal_flag = clean_label(row.get("malignant_label_mapped", None))  # "yes" or "no" or "not_applicable"

    for head in profile_cols:
        # Branch consistency guard
        w = HEAD_W.get(head, DEFAULT_W)
        if w == 0:
            continue
        if mal_flag == "yes" and head == "non_malignant_subtype":
            continue
        if mal_flag == "no" and head == "malignant_subtype":
            continue

        probs_head = head_to_probs[head]
        class_index = head_to_class_index[head]

        label1 = clean_label(row.get(head + "_mapped", None))
        if label1 is None:
            continue

        # Subtype handling: allow 2 labels (primary OR second), take max prob once
        if head == "malignant_subtype":
            label2 = clean_label(row.get("second_malignant_subtype_mapped", None))

            p1 = probs_head[:, class_index[label1]] if label1 in class_index else None
            p2 = probs_head[:, class_index[label2]] if (label2 and label2 in class_index) else None

            if p1 is None and p2 is None:
                continue
            if p1 is None:
                p = p2
            elif p2 is None:
                p = p1
            else:
                p = np.maximum(p1, p2)

            score += w * safe_log(p)           
            continue

        if head == "non_malignant_subtype":
            label2 = clean_label(row.get("second_non_malignant_subtype_mapped", None))

            p1 = probs_head[:, class_index[label1]] if label1 in class_index else None
            p2 = probs_head[:, class_index[label2]] if (label2 and label2 in class_index) else None

            if p1 is None and p2 is None:
                continue
            if p1 is None:
                p = p2
            elif p2 is None:
                p = p1
            else:
                p = np.maximum(p1, p2)

            score += w * safe_log(p)
            continue

        # Normal case: single label
        if label1 not in class_index:
            continue

        score += w * safe_log(probs_head[:, class_index[label1]])

    log_scores[:, hpc_to_col[hpc]] = score

print("log_scores min max:", float(log_scores.min()), float(log_scores.max()))

# 9) Convert scores to per tile probability over HPC ids
T_FINAL= 5.0  


# Diagnostics: do NOT overwrite the export probs
for T in [1.0, 1.5, 2.0, 3.0, 5.0, 8.0]:
    probs_T = softmax_np(log_scores / T, axis=1)
    top_idx_T = np.argmax(probs_T, axis=1)
    top_prob_T = probs_T[np.arange(n_tiles), top_idx_T]
    print("T", T, "mean top prob", float(top_prob_T.mean()), "median", float(np.median(top_prob_T)))

# Export: compute once using the final temperature
hpc_probs = softmax_np(log_scores / T_FINAL, axis=1)
top_idx = np.argmax(hpc_probs, axis=1)
top_hpc = [hpc_ids[i] for i in top_idx]
top_prob = hpc_probs[np.arange(n_tiles), top_idx]

# 10) Build final wide table for Streamlit
keep_cols = [c for c in ["slides", "tiles", "slide_tile", "x_native", "y_native"] if c in df_heatmap.columns]
out = df_heatmap[keep_cols].copy()

out["hpc_top1"] = top_hpc
out["hpc_top1_prob"] = top_prob

for j, hpc in enumerate(hpc_ids):
    out[f"p_hpc_{hpc}"] = hpc_probs[:, j]

OUT_CSV = "/Users/vaidehipandya/Desktop/Work/Heatmap/tile_hpc_probabilities_wide.csv"
out.to_csv(OUT_CSV, index=False)

print("Saved:", OUT_CSV)
print(out.head(5))

# Optional sanity check: probabilities should sum to ~1
prob_cols = [c for c in out.columns if c.startswith("p_hpc_")]
print("Max deviation from 1:", float((out[prob_cols].sum(axis=1) - 1).abs().max()))

Profile cols used: ['cluster_homogeneity', 'malignant_label', 'inflammation_label', 'necrosis_label', 'stromal_epithelium_ratio', 'stromal_cellularity', 'malignant_subtype', 'non_malignant_subtype']
HPCs after filtering: 71
Tiles: 254 HPCs scored: 71
log_scores min max: -67.44914245605469 0.0
T 1.0 mean top prob 0.46756985783576965 median 0.3333304524421692
T 1.5 mean top prob 0.40639689564704895 median 0.3278847634792328
T 2.0 mean top prob 0.35425642132759094 median 0.2897099256515503
T 3.0 mean top prob 0.2716712951660156 median 0.19956228137016296
T 5.0 mean top prob 0.16854849457740784 median 0.11127953976392746
T 8.0 mean top prob 0.09683388471603394 median 0.07041779160499573
Saved: /Users/vaidehipandya/Desktop/Work/Heatmap/tile_hpc_probabilities_wide.csv
                    slides                          slide_tile  x_native  \
0  TCGA-55-7574-01Z-00-DX1  TCGA-55-7574-01Z-00-DX1_18_15.jpeg     28800   
1  TCGA-55-7574-01Z-00-DX1   TCGA-55-7574-01Z-00-DX1_14_6.jpeg     22400   